# Document Classification Pipeline - Orchestration Verification

This notebook demonstrates the simplified orchestration of the EKB pipeline using the new `run()` method.

In [1]:
import sys
import os
from google.cloud import storage

# Ensure repo root is in path
repo_root = os.path.abspath(os.path.join(os.getcwd(), "../../.."))
if repo_root not in sys.path:
    sys.path.append(repo_root)

from pipelines.enterprise_knowledge_base import ClassificationPipeline

print("Services imported.")

Services imported.


## 1. Ingestion Helper
Upload a file to the landing zone with metadata.

In [2]:
def ingest_local_file(local_path: str, metadata: dict) -> str:
    client = storage.Client()
    bucket_name = "enterprise_knowledgebase_landing_zone"
    filename = os.path.basename(local_path)
    bucket = client.bucket(bucket_name)
    blob = bucket.blob(filename)
    blob.metadata = metadata
    blob.upload_from_filename(local_path)
    return f"gs://{bucket_name}/{filename}"

## 2. Execute Pipeline Orchestration
Using the single `run()` method.

In [ ]:
pipeline = ClassificationPipeline()

# 1. Ingest
local_path = "../../../BH1132462.pdf"
metadata = {
    "project": "orchestration-test",
    "domain": "it",
    "trust-level": "wip",
    "uploader": "tester@example.com"
}

if os.path.exists(local_path):
    landing_uri = ingest_local_file(local_path, metadata)
    print(f"Ingested: {landing_uri}")

    # 2. Run Orchestration
    print("Starting Pipeline Orchestration...")
    result = pipeline.run(landing_uri)

    print("\n--- Pipeline Result ---")
    print(f"Final Domain: {result.final_domain}")
    print(f"Security Tier: {result.final_security_tier}")
    print(f"Sanitized URI: {result.final_sanitized_uri}")
else:
    print("Sample file not found.")

2026-04-24 17:17:36.358 | INFO     | pipelines.enterprise_knowledge_base.document_classification.pipeline:run:54 - Starting pipeline orchestration for: gs://enterprise_knowledgebase_landing_zone/PASAPORTE.pdf
2026-04-24 17:17:36.359 | DEBUG    | pipelines.enterprise_knowledge_base.document_classification.pipeline:_get_blob_metadata:188 - Extracting metadata for: gs://enterprise_knowledgebase_landing_zone/PASAPORTE.pdf
2026-04-24 17:17:36.359 | INFO     | pipelines.enterprise_knowledge_base.document_classification.gcs_service.service:get_blob_metadata:31 - Extracting detailed GCS metadata for: gs://enterprise_knowledgebase_landing_zone/PASAPORTE.pdf
2026-04-24 17:17:36.360 | DEBUG    | pipelines.enterprise_knowledge_base.document_classification.gcs_service.service:_parse_uri:132 - Parsing GCS URI into dictionary: gs://enterprise_knowledgebase_landing_zone/PASAPORTE.pdf


Ingested: gs://enterprise_knowledgebase_landing_zone/PASAPORTE.pdf
Starting Pipeline Orchestration...


2026-04-24 17:17:36.709 | INFO     | pipelines.enterprise_knowledge_base.document_classification.pipeline:dlp_trigger:156 - Triggering DLP scan for: gs://enterprise_knowledgebase_landing_zone/PASAPORTE.pdf
2026-04-24 17:17:36.709 | INFO     | pipelines.enterprise_knowledge_base.document_classification.dlp_service.service:inspect_gcs_file:38 - Starting DLP scan for: gs://enterprise_knowledgebase_landing_zone/PASAPORTE.pdf
2026-04-24 17:17:36.710 | DEBUG    | pipelines.enterprise_knowledge_base.document_classification.dlp_service.service:_get_custom_keywords_config:209 - Building custom keywords config for inspection.
2026-04-24 17:17:37.629 | INFO     | pipelines.enterprise_knowledge_base.document_classification.dlp_service.service:inspect_gcs_file:66 - DLP Job created: projects/ag-core-dev-fdx7/locations/global/dlpJobs/i-6036280318270609215
2026-04-24 17:17:37.750 | INFO     | pipelines.enterprise_knowledge_base.document_classification.dlp_service.service:wait_for_job:104 - Waiting for